# Final Training — Load Best Params from Optuna

Загружает best_params из Optuna study (сохранено в rnn_otune.ipynb),
затем делает финальное обучение на 100 эпохах с этими параметрами.
Экспортирует ONNX модель и генерирует RNN_Scaler.mqh для EA.


In [ ]:
import os, glob, io, json, sys, math, hashlib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import onnx
import onnxruntime as ort
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
RNN_DIR = Path(os.environ.get('RNN_DIR', '/content/drive/MyDrive/RNN'))
ARTIFACTS_DIR = RNN_DIR / 'data_normalized'
OUTPUT_DIR = RNN_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_FEATURES, N_BARS, N_CLASSES = 12, 20, 2
BATCH_SIZE, FINAL_EPOCHS = 64, 40
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load Best Params from Optuna

In [ ]:
# Р—Р°РіСЂСѓР¶Р°РµРј best_params.json СЃРѕС…СЂР°РЅРµРЅРЅС‹Р№ РІ rnn_learn_v3_improved.ipynb
best_params_file = OUTPUT_DIR / 'best_params.json'

if not best_params_file.exists():
    raise FileNotFoundError(f'best_params.json not found: {best_params_file}')

with open(best_params_file, 'r') as f:
    optuna_results = json.load(f)

if optuna_results.get('input_contract') != 'oldest_to_newest_last_seq_v1':
    raise ValueError('Run the updated Optuna notebook before final training')
buy_params = optuna_results['best_params']
buy_metrics_optuna = optuna_results['metrics']
best_trial_num = optuna_results['best_trial_number']

print(f"Loaded Optuna results from Trial {best_trial_num}:")
print(f"  Precision: {optuna_results['best_value']:.4f}")
print(f"  TP: {buy_metrics_optuna['tp']}, FP: {buy_metrics_optuna['fp']}")
print(f"\nBest Parameters:")
for k, v in buy_params.items():
    print(f"  {k}: {v}")

## 2. Load Data + Model

In [ ]:
def load_npz(filepath, seq_len):
    """Load NPZ Рё СЂР°Р·РґРµР»РёС‚СЊ РЅР° train/test."""
    data = np.load(filepath, allow_pickle=True)
    X = data['X'].astype(np.float32)
    if X.ndim != 3 or X.shape[1:] != (N_BARS, N_FEATURES) or not np.isfinite(X).all():
        raise ValueError('Expected finite normalized [N, 20, 12] data')
    y = data['y'].astype(np.int64)
    n_total = len(X)
    actual_seq = min(seq_len, N_BARS)
    X = X[:, -actual_seq:, :]  # oldest to newest, includes signal bar
    n_train = int(n_total * 0.8)
    X_train, X_test = X[:n_train], X[n_train:]
    y_train, y_test = y[:n_train], y[n_train:]
    num_classes = max(2, len(np.unique(y)))
    y_train_oh = np.eye(num_classes)[y_train]
    y_test_oh  = np.eye(num_classes)[y_test]
    print(f'  rows={n_total}, train={n_train}, test={n_total-n_train}, X={X.shape}')
    print(f'  classes: {dict(zip(*np.unique(y, return_counts=True)))}')
    train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train_oh))
    test_ds  = TensorDataset(torch.FloatTensor(X_test),  torch.FloatTensor(y_test_oh))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, test_loader, actual_seq

# Р—Р°РіСЂСѓР¶Р°РµРј РґР°РЅРЅС‹Рµ
buy_npz = sorted(glob.glob(str(ARTIFACTS_DIR / 'data_BUY_*_normalized.npz')))
if not buy_npz:
    raise FileNotFoundError(f'No BUY NPZ in {ARTIFACTS_DIR}/')

train_loader, test_loader, seq_len_actual = load_npz(buy_npz[0], buy_params['seq_len'])
print(f"\nSeq len: {seq_len_actual}")

In [4]:
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        out, _ = self.gru(x, h0)
        out = out[:, -1, :]
        out = self.dropout(out)
        return self.fc(out)

# РЎРѕР·РґР°РµРј РјРѕРґРµР»СЊ СЃ best РїР°СЂР°РјРµС‚СЂР°РјРё
model = GRUClassifier(
    N_FEATURES,
    buy_params['hidden_size'],
    buy_params['num_layers'],
    N_CLASSES,
    buy_params['dropout']
)
print(f"Model created: {model}")

Model created: GRUClassifier(
  (gru): GRU(12, 112, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=112, out_features=2, bias=True)
)


## 3. Final Training

In [5]:
def train_final(model, train_loader, epochs, lr, device, verbose=True):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.1)
    best_acc = 0.0
    best_state = None

    it = tqdm(range(epochs), desc='Final Train', leave=True) if verbose else range(epochs)
    for epoch in it:
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch.argmax(dim=1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * X_batch.size(0)
            correct   += (y_pred.argmax(dim=1) == y_batch.argmax(dim=1)).sum().item()
            total     += X_batch.size(0)
        train_loss = total_loss / total
        train_acc  = correct / total
        scheduler.step()
        if train_acc > best_acc:
            best_acc = train_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose and hasattr(it, 'set_postfix'):
            it.set_postfix({'loss': f'{train_loss:.4f}', 'acc': f'{train_acc:.4f}'})
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_acc, train_loss

# Р¤РёРЅР°Р»СЊРЅРѕРµ РѕР±СѓС‡РµРЅРёРµ
print(f"\nFinal Training with {FINAL_EPOCHS} epochs")
print(f"  Learning rate: {buy_params['lr']}")
print(f"  Dropout: {buy_params['dropout']}")
print()
model, train_acc, train_loss = train_final(model, train_loader, FINAL_EPOCHS, buy_params['lr'], DEVICE, verbose=True)
print(f"\nFinal train accuracy: {train_acc:.4f}, loss: {train_loss:.4f}")


Final Training with 100 epochs
  Learning rate: 0.003283136695374537
  Dropout: 0.2



Final Train:   0%|          | 0/100 [00:00<?, ?it/s]


Final train accuracy: 0.9723, loss: 0.0561


## 4. Export ONNX + RNN_Scaler.mqh

In [ ]:
# A stale result must never authorize a new or failed export.
(OUTPUT_DIR / 'final_results.json').unlink(missing_ok=True)

def export_to_onnx(model, seq_len, n_features, out_path, samples):
    model.eval().to('cpu')
    if samples.ndim != 3 or samples.shape[1:] != (seq_len, n_features) or not np.isfinite(samples).all():
        raise ValueError('Invalid or non-finite parity samples')
    tmp = str(out_path) + '.tmp'
    try:
        dummy = torch.from_numpy(samples[:1])
        torch.onnx.export(model, dummy, tmp, input_names=['input'], output_names=['output'],
                          dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
                          opset_version=18)
        graph = onnx.load(tmp, load_external_data=False)
        if any(t.data_location == onnx.TensorProto.EXTERNAL for t in graph.graph.initializer):
            raise ValueError('ONNX external data is not supported')
        onnx.checker.check_model(graph)
        if len(graph.graph.input) != 1 or len(graph.graph.output) != 1:
            raise ValueError('ONNX must have exactly one input and one output')
        inp = graph.graph.input[0].type.tensor_type
        dims = inp.shape.dim
        if inp.elem_type != onnx.TensorProto.FLOAT or len(dims) != 3 or dims[1].dim_value != seq_len or dims[2].dim_value != n_features:
            raise ValueError('ONNX input must be float32 [batch, seq_len, 12]')
        session = ort.InferenceSession(tmp, providers=['CPUExecutionProvider'])
        atol, rtol = 1e-5, 1e-4
        max_diff = 0.0
        with torch.no_grad():
            for sample in samples:
                x = np.ascontiguousarray(sample[None, :, :], dtype=np.float32)
                expected = model(torch.from_numpy(x)).cpu().numpy()
                actual = session.run(None, {session.get_inputs()[0].name: x})[0]
                if expected.shape != (1, 2) or actual.shape != (1, 2):
                    raise ValueError('Model output must be logits [1, 2]')
                if not np.isfinite(expected).all() or not np.isfinite(actual).all():
                    raise ValueError('Non-finite model output')
                max_diff = max(max_diff, float(np.max(np.abs(expected - actual))))
                if not np.allclose(expected, actual, atol=atol, rtol=rtol):
                    raise ValueError(f'PyTorch/ONNX mismatch: {expected} vs {actual}')
        out_path.write_bytes(graph.SerializeToString())
        return {'passed': True, 'samples': len(samples), 'atol': atol,
                'rtol': rtol, 'max_abs_diff': max_diff}
    finally:
        for filename in (tmp, tmp + '.data'):
            if os.path.exists(filename):
                os.remove(filename)

def generate_scaler_mqh(seq_len, n_features, eps, norm_window, model_hash, out_path):
    text = f'''// Generated after verified final training. Do not edit.
#ifndef RNN_SCALER_MQH
#define RNN_SCALER_MQH
#define NN_BUY_MODEL_SHA256 "{model_hash}"
#define NN_BUY_SEQ_LEN {seq_len}
#define NN_SELL_SEQ_LEN {seq_len}
#define NN_MAX_SEQ_LEN {seq_len}
#define NN_FEATURES {n_features}
#define NN_NORM_STD_EPS {eps:g}
#define NN_BUY_NORM_WINDOW {norm_window}
#define NN_SELL_NORM_WINDOW {norm_window}
#define NN_MAX_NORM_WINDOW {norm_window}
#define NN_SEQ_LEN NN_BUY_SEQ_LEN
#endif
'''
    out_path.write_text(text, encoding='utf-8')

onnx_dir = OUTPUT_DIR / 'onnx'
onnx_dir.mkdir(exist_ok=True)
model_path = onnx_dir / 'buy_model.onnx'
# Holdout examples are deterministic and do not depend on shuffled loaders.
samples = test_loader.dataset.tensors[0][:32].numpy().astype(np.float32)
if len(samples) == 0:
    raise ValueError('A non-empty holdout set is required for ONNX validation')
parity = export_to_onnx(model, seq_len_actual, N_FEATURES, model_path, samples)
model_sha256 = hashlib.sha256(model_path.read_bytes()).hexdigest()
generate_scaler_mqh(seq_len_actual, N_FEATURES, 1e-6, N_BARS, model_sha256, OUTPUT_DIR / 'RNN_Scaler.mqh')
models_dir = OUTPUT_DIR / 'models'
models_dir.mkdir(exist_ok=True)
torch.save(model.state_dict(), models_dir / 'model_buy_final.pth')
print(f'PyTorch/ONNX verified on {parity["samples"]} examples; max error={parity["max_abs_diff"]:.3g}')


## 5. Save Final Results

In [ ]:
# РЎРѕС…СЂР°РЅСЏРµРј С„РёРЅР°Р»СЊРЅС‹Рµ СЂРµР·СѓР»СЊС‚Р°С‚С‹
final_summary = {
    'input_contract': 'oldest_to_newest_last_seq_v1',
    'model_sha256': model_sha256,
    'onnx_parity': parity,
    'timestamp': datetime.now().isoformat(),
    'optuna_trial_number': best_trial_num,
    'optuna_precision': optuna_results['best_value'],
    'optuna_metrics': buy_metrics_optuna,
    'best_params': buy_params,
    'final_training': {
        'epochs': FINAL_EPOCHS,
        'train_acc': float(train_acc),
        'train_loss': float(train_loss)
    },
    'model_config': {
        'n_features': N_FEATURES,
        'n_bars': N_BARS,
        'seq_len': seq_len_actual,
    }
}

with open(OUTPUT_DIR / 'final_results.json', 'w') as f:
    json.dump(final_summary, f, indent=2)

print(f"\n{'='*70}")
print(f"FINAL TRAINING COMPLETE")
print(f"{'='*70}")
print(f"\nFinal Results:")
print(f"  Train Accuracy: {train_acc:.4f}")
print(f"  Train Loss: {train_loss:.4f}")
print(f"\nOptuna Baseline (Trial {best_trial_num}):")
print(f"  Precision: {optuna_results['best_value']:.4f}")
print(f"  TP: {buy_metrics_optuna['tp']}, FP: {buy_metrics_optuna['fp']}")
print(f"\nFiles for MT5:")
print(f"  1. {onnx_dir / 'buy_model.onnx'} -> Common\\Files\\RNN_GOLD_M5\\buy_model.onnx")
print(f"  2. {OUTPUT_DIR / 'RNN_Scaler.mqh'} -> MQL5\\Experts\\RNN_gold_M5_BUY\\RNN_Scaler.mqh")
print(f"\nRun python training/deploy_rnn_model.py to validate and copy files to MT5.")